In [1]:
# # Cell 1 — Install required packages (run once)
# # If running in an environment where packages are already installed, you can skip this cell.
# %pip install -q transformers datasets evaluate scikit-learn pandas tqdm


In [2]:
# %pip uninstall -y torch torchvision torchaudio
# %pip cache purge
# %pip install --index-url https://download.pytorch.org/whl/cpu torch torchvision torchaudio


In [3]:
import sys, os
print("Python:", sys.executable, sys.version)
print("CWD files:", os.listdir('.'))
print("sys.path first entries:", sys.path[:5])
# Does a local file named 'torch.py' exist?
print("'torch.py' in CWD?", os.path.exists('torch.py'))
# Show any installed torch package info (won't import torch)
import importlib.util
print("torch spec:", importlib.util.find_spec("torch"))


Python: c:\Users\94772\AppData\Local\Programs\Python\Python310\python.exe 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]
CWD files: ['Acadamic.ipynb', 'AI MOBILE APP FOR EARLY IDENTIFICATION OF STUDENT MENTAL HEALTH VIA BEHAVIOR full.pdf', 'Backend', 'burnout2classes.ipynb', 'BurnoutNLP.ipynb', 'burnout_text_classifier_ft', 'Data', 'Data EVOL MH.csv', 'data_with_burnout.csv', 'mental_health_dataset.xlsx', 'nlp_burnout_demo', 'saved_models_binary_high_v2_all', 'saved_models_binary_high_v2_cleaned', 'synthetic_academic_dataset_v2.csv', 'synthetic_reflective_journals_single.csv']
sys.path first entries: ['c:\\Users\\94772\\AppData\\Local\\Programs\\Python\\Python310\\python310.zip', 'c:\\Users\\94772\\AppData\\Local\\Programs\\Python\\Python310\\DLLs', 'c:\\Users\\94772\\AppData\\Local\\Programs\\Python\\Python310\\lib', 'c:\\Users\\94772\\AppData\\Local\\Programs\\Python\\Python310', '']
'torch.py' in CWD? False
torch spec: ModuleSpec(name='torch', loade

In [4]:
# # In a notebook cell:
# !pip install -q vaderSentiment


In [5]:
# %pip uninstall -y keras keras-nightly keras-cpu


In [6]:
# %pip uninstall -y tensorflow tensorflow-intel tensorflow-cpu tensorflow-gpu


In [7]:
# %pip install transformers

In [8]:
# %pip install accelerate sentence-transformers datasets evaluate
# %pip install vaderSentiment


In [9]:
# Cell 2 — Imports, reproducibility, device
import os, random
import pandas as pd
import numpy as np
import torch
from typing import List

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, pipeline)
from datasets import Dataset, DatasetDict
import evaluate
from sentence_transformers import SentenceTransformer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cpu


In [10]:
# Cell 3 — Create a single synthetic reflective-journal CSV (no split yet)
# We create unlabeled-style varied realistic journal entries, then create labels for supervised part.
# In real life: you will collect journals and label with MBI/CBI or expert annotation.

journal_texts = [
    "I feel exhausted all the time and nothing I do makes a difference.",
    "I can't focus on my assignments, everything feels pointless lately.",
    "I slept poorly and still felt drained after waking up today.",
    "I am overwhelmed by deadlines and can't seem to start anything.",
    "I enjoy studying with friends and felt motivated this week.",
    "I managed to finish projects and feel proud and energetic.",
    "I had a busy but productive week and I'm optimistic.",
    "My motivation dropped; I attend class but I feel numb.",
    "I made many mistakes and now I feel like a failure.",
    "I am doing small breaks and it helps me relax and study."
]

# Create synthetic labels: 1=burnout, 0=no_burnout
# For demo: mark clearly negative items as burnout
labels = [1,1,1,1,0,0,0,1,1,0]

# Build dataframe and save CSV
df = pd.DataFrame({"text": journal_texts, "label": labels})
df.to_csv("synthetic_reflective_journals_single.csv", index=False)
print("Saved synthetic CSV with", len(df), "rows -> synthetic_reflective_journals_single.csv")
df.head()


Saved synthetic CSV with 10 rows -> synthetic_reflective_journals_single.csv


,text,label
0,I feel exhausted all the time and nothing I do...,1
1,"I can't focus on my assignments, everything fe...",1
2,I slept poorly and still felt drained after wa...,1
3,I am overwhelmed by deadlines and can't seem t...,1
4,I enjoy studying with friends and felt motivat...,0


In [11]:
# Cell 4 — Show dataset and a quick class balance
df = pd.read_csv("synthetic_reflective_journals_single.csv")
print(df['label'].value_counts())
df.sample(5, random_state=RANDOM_SEED)


label
1    6
0    4
Name: count, dtype: int64


,text,label
8,I made many mistakes and now I feel like a fai...,1
1,"I can't focus on my assignments, everything fe...",1
5,I managed to finish projects and feel proud an...,0
0,I feel exhausted all the time and nothing I do...,1
7,My motivation dropped; I attend class but I fe...,1


In [12]:
# Cell 5 — Prepare datasets for supervised fine-tuning (train/val/test splits)
# Convert dataframe to Hugging Face Dataset
ds = Dataset.from_pandas(df)
# Use 60% train, 20% val, 20% test (small demo)
train_test = ds.train_test_split(test_size=0.4, seed=RANDOM_SEED)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=RANDOM_SEED)
dataset_dict = DatasetDict({
    'train': train_test['train'],
    'validation': val_test['train'],
    'test': val_test['test']
})
print(dataset_dict)


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 6
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2
    })
})


In [13]:
# Cell 6 — Tokenizer + preprocessing for supervised model (DistilBERT)
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding="max_length", max_length=128)

tokenized = dataset_dict.map(preprocess_function, batched=True)
tokenized = tokenized.remove_columns(["text"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")
print("Tokenized datasets ready.")


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenized datasets ready.


In [14]:
# # In a notebook cell:
# !pip install -q "accelerate>=0.26.0"
# # or install transformers with torch extras (includes accelerate)
# !pip install -q "transformers[torch]"


In [15]:
# Cell 7 — Create model and Trainer (supervised)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

# Metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'f1': f1.compute(predictions=preds, references=labels, average='weighted')['f1']
    }

training_args = TrainingArguments(
    output_dir="nlp_burnout_demo",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    seed=RANDOM_SEED,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
print("Trainer created.")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\94772\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trainer created.


In [16]:
# Cell 8 — Train supervised model (DistilBERT fine-tuning)
# NOTE: On CPU this will be slow but okay for tiny synthetic data. On GPU it's fast.
train_result = trainer.train()
metrics = trainer.evaluate(tokenized['test'])
print("Test metrics:", metrics)
# Save fine-tuned model
trainer.save_model("burnout_text_classifier_ft")
tokenizer.save_pretrained("burnout_text_classifier_ft")
print("Saved fine-tuned model to burnout_text_classifier_ft/")


  0%|          | 0/6 [00:00<?, ?it/s]

c:\Users\94772\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.7139427065849304, 'eval_accuracy': 0.0, 'eval_f1': 0.0, 'eval_runtime': 0.189, 'eval_samples_per_second': 10.58, 'eval_steps_per_second': 5.29, 'epoch': 1.0}


c:\Users\94772\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


...

  0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.7674745321273804, 'eval_accuracy': 0.0, 'eval_f1': 0.0, 'eval_runtime': 0.132, 'eval_samples_per_second': 15.155, 'eval_steps_per_second': 7.577, 'epoch': 3.0}
{'train_runtime': 9.3512, 'train_samples_per_second': 1.925, 'train_steps_per_second': 0.642, 'train_loss': 0.632607618967692, 'epoch': 3.0}


c:\Users\94772\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/1 [00:00<?, ?it/s]

Test metrics: {'eval_loss': 0.6530634164810181, 'eval_accuracy': 1.0, 'eval_f1': 1.0, 'eval_runtime': 0.1334, 'eval_samples_per_second': 14.994, 'eval_steps_per_second': 7.497, 'epoch': 3.0}
Saved fine-tuned model to burnout_text_classifier_ft/


In [17]:
# Cell 9 — Create hybrid single-journal detector components (zero-shot, embeddings, sentiment, lexicon, anomaly)
# Zero-shot classifier (NLI)
zs_pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0 if device=="cuda" else -1)

# Sentence Transformer for embeddings (used for anomaly detection)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# VADER sentiment for quick sentiment features
vader = SentimentIntensityAnalyzer()

# Build a small "normal" corpus embeddings for demo (in production: populate with many normal journal entries)
normal_corpus = [
    "I had a productive week and feel good about my studies.",
    "I slept well and managed to study, feeling motivated.",
    "I exercised and that improved my focus today.",
    "I finished my projects and felt satisfied and energetic.",
    "I get some stress but I can manage with short breaks."
]
normal_embs = embed_model.encode(normal_corpus)
iso_forest = IsolationForest(contamination=0.1, random_state=RANDOM_SEED).fit(normal_embs)
print("Hybrid components ready (zero-shot, embed, iso, vader).")


Hybrid components ready (zero-shot, embed, iso, vader).


In [18]:
# Cell 10 — Lexicon for burnout-features and helper functions
burnout_keywords = [
    "exhaust", "drain", "detached", "overwhelm", "fail", "can't focus", "no motivation",
    "empty", "hopeless", "burnout", "numb", "worthless", "can't", "pointless", "stressed"
]

def lexicon_score(text):
    t = text.lower()
    count = sum(1 for kw in burnout_keywords if kw in t)
    # normalize assuming ~5 matches max for demo
    return min(1.0, count / 5.0)

def anomaly_score(text):
    emb = embed_model.encode([text])
    raw = -iso_forest.score_samples(emb)[0]  # higher -> more anomalous
    # map to 0-1 roughly via logistic
    return float(1/(1+np.exp(- (raw - 0.5) )))

def sentiment_compound_negative(text):
    s = vader.polarity_scores(text)
    # Use negative polarity magnitude (if compound<0)
    return float(abs(min(0.0, s['compound'])))


In [19]:
# Cell 11 — Single-entry inference function (combines supervised model if available, else zero-shot)
# Outputs: detailed component scores and final risk_score (0-1) and label (low/medium/high)

# try to load fine-tuned transformer via pipeline
try:
    supervised_pipe = pipeline("text-classification", model="burnout_text_classifier_ft", tokenizer="burnout_text_classifier_ft", device=0 if device=="cuda" else -1)
    supervised_available = True
    print("Loaded supervised fine-tuned pipeline.")
except Exception as e:
    supervised_pipe = None
    supervised_available = False
    print("Supervised model not available (fallback to zero-shot).", str(e))

def predict_single_journal(text: str, weights=None):
    """Return dict with components and final risk label"""
    if weights is None:
        # default weights (tune later with real data)
        weights = {
            "supervised": 0.5,   # if available, supervised model is most trusted
            "zero_shot": 0.3,    # fallback if supervised absent
            "anomaly": 0.1,
            "lexicon": 0.07,
            "sent_neg": 0.03
        }
    # supervised prob
    if supervised_available:
        sup_out = supervised_pipe(text, truncation=True, max_length=128)[0]
        # labels might be 'LABEL_0' or 'LABEL_1' or '0'/'1' or 'BURNOUT' depending on config.
        # We convert label->prob of burnout by checking mapping via model config if possible.
        # For our saved setup, label 'LABEL_1' corresponds to index 1 -> burnout in training
        lab = sup_out['label']
        score = sup_out['score']
        # heuristic: if label contains '1' or 'LABEL_1' or 'LABEL_0' mapping uncertain; fallback: use score for label 'LABEL_1'
        if lab.lower().endswith("1") or lab.lower().startswith("label_1") or lab == "LABEL_1":
            supervised_prob = score
        elif lab.lower().endswith("0") or lab.lower().startswith("label_0") or lab == "LABEL_0":
            supervised_prob = 1.0 - score
        else:
            # if label is 'BURNOUT' or 'NO_BURNOUT' detect
            if "burnout" in lab.lower():
                supervised_prob = score
            else:
                supervised_prob = 1.0 - score
    else:
        supervised_prob = None

    # zero-shot
    zs_out = zs_pipe(text, candidate_labels=["burnout", "no_burnout"], multi_label=False)
    # find burnout score
    try:
        idx = zs_out['labels'].index("burnout")
        zs_prob = zs_out['scores'][idx]
    except ValueError:
        zs_prob = zs_out['scores'][0]

    # anomaly, lexicon, sentiment
    anom = anomaly_score(text)
    lex = lexicon_score(text)
    sent_neg = sentiment_compound_negative(text)

    # combine
    if supervised_prob is not None:
        score = (weights['supervised'] * supervised_prob
                 + weights['anomaly'] * anom
                 + weights['lexicon'] * lex
                 + weights['sent_neg'] * sent_neg)
    else:
        # fallback combining zero-shot and others
        score = (weights['zero_shot'] * zs_prob
                 + weights['anomaly'] * anom
                 + weights['lexicon'] * lex
                 + weights['sent_neg'] * sent_neg)

    # normalize to [0,1]
    score = float(max(0.0, min(1.0, score)))

    if score > 0.7:
        label = "high_risk"
    elif score > 0.4:
        label = "medium_risk"
    else:
        label = "low_risk"

    return {
        "text": text,
        "supervised_prob": supervised_prob,
        "zero_shot_prob": zs_prob,
        "anomaly_score": anom,
        "lexicon_score": lex,
        "sentiment_negative": sent_neg,
        "risk_score": score,
        "label": label,
        "zero_shot_raw": zs_out
    }


Loaded supervised fine-tuned pipeline.


In [20]:
# Cell 12 — Demo: run several single-journal predictions
test_entries = [
    "I feel exhausted all the time and nothing I do makes a difference.",
    "I slept well and feel ready to tackle my assignments with energy.",
    "I go to classes but feel numb, and nothing motivates me to study.",
    "I had a stressful week but took breaks and felt better afterwards."
]

for t in test_entries:
    out = predict_single_journal(t)
    print("-"*80)
    print("Text:", t)
    print("Risk score: {:.3f}  Label: {}".format(out['risk_score'], out['label']))
    print("Components -> supervised_prob:", out['supervised_prob'],
          "zero_shot:", round(out['zero_shot_prob'],3),
          "anomaly:", round(out['anomaly_score'],3),
          "lexicon:", round(out['lexicon_score'],3),
          "sent_neg:", round(out['sentiment_negative'],3))


--------------------------------------------------------------------------------
Text: I feel exhausted all the time and nothing I do makes a difference.
Risk score: 0.341  Label: low_risk
Components -> supervised_prob: 0.53187096118927 zero_shot: 0.724 anomaly: 0.497 lexicon: 0.2 sent_neg: 0.361
--------------------------------------------------------------------------------
Text: I slept well and feel ready to tackle my assignments with energy.
Risk score: 0.303  Label: low_risk
Components -> supervised_prob: 0.5071746706962585 zero_shot: 0.064 anomaly: 0.495 lexicon: 0.0 sent_neg: 0.0
--------------------------------------------------------------------------------
Text: I go to classes but feel numb, and nothing motivates me to study.
Risk score: 0.333  Label: low_risk
Components -> supervised_prob: 0.5111509561538696 zero_shot: 0.377 anomaly: 0.494 lexicon: 0.2 sent_neg: 0.477
--------------------------------------------------------------------------------
Text: I had a stressful w

In [21]:
# Cell 13 — How to replace synthetic CSV with your real data (instructions & small helper)
print("To use real data: prepare CSV with columns 'text' and 'label' (label optional for supervised fine-tuning).")
print("Example: df_real = pd.read_csv('your_journals.csv') and then convert to Hugging Face Dataset and follow tokenization/training steps.")
print("""
If you have real labeled data:
1) Replace the CSV loaded earlier.
2) Recreate dataset_dict with train/val/test splits (or provide separate files).
3) Re-run tokenization (Cell 6), trainer (Cell 7) and training (Cell 8).
4) Re-run Cell 11 to use supervised model predictions (the code will detect the saved model).
""")


To use real data: prepare CSV with columns 'text' and 'label' (label optional for supervised fine-tuning).
Example: df_real = pd.read_csv('your_journals.csv') and then convert to Hugging Face Dataset and follow tokenization/training steps.

If you have real labeled data:
1) Replace the CSV loaded earlier.
2) Recreate dataset_dict with train/val/test splits (or provide separate files).
3) Re-run tokenization (Cell 6), trainer (Cell 7) and training (Cell 8).
4) Re-run Cell 11 to use supervised model predictions (the code will detect the saved model).



In [22]:
# Cell 14 — Quick note: integrate with behavior model later
print("""
Integration notes for the full proposal:
- Export supervised_prob (or zero-shot prob when supervised unavailable) from this module as a numeric feature.
- Combine it with behavioral model outputs (e.g., RF/XGBoost probabilities) in a fusion model (late fusion).
- Fusion output => severity level (low/medium/high) -> feed to intervention module (policy rules or RL agent).
""")



Integration notes for the full proposal:
- Export supervised_prob (or zero-shot prob when supervised unavailable) from this module as a numeric feature.
- Combine it with behavioral model outputs (e.g., RF/XGBoost probabilities) in a fusion model (late fusion).
- Fusion output => severity level (low/medium/high) -> feed to intervention module (policy rules or RL agent).



In [23]:
# Cell 15 — Save notebook artifacts & test CSV download (optional)
# Confirm files exist
print("Files saved in working dir:", [f for f in os.listdir('.') if f.startswith('synthetic_reflective') or f.startswith('burnout_text_classifier')])


Files saved in working dir: ['burnout_text_classifier_ft', 'synthetic_reflective_journals_single.csv']


In [24]:
trainer.save_model("burnout_text_classifier_ft")


SafetensorError: Error while serializing: IoError(Os { code: 1224, kind: Uncategorized, message: "The requested operation cannot be performed on a file with a user-mapped section open." })